In [1]:
import os
import sys
import json
import time
from pathlib import Path

from typing import List, Literal
from pydantic import BaseModel, Field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from util.text_similarity import rank_texts

from openai import OpenAI

from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", None)  # show all rows
pd.set_option("display.width", None)  # no line wrap
pd.set_option("display.max_colwidth", None)  # full column text


In [ ]:
client = OpenAI()

In [3]:
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ARTIFACT_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/artifacts"
annotation_counts_df = pd.read_csv(ARTIFACT_OUTPUT_DIR / "annotation_counts.csv")
display(annotation_counts_df.head())

,annotation,count
0,hypothetical protein,10582687
1,MULTISPECIES: hypothetical protein,1478845
2,ABC transporter ATP-binding protein,826537
3,ABC transporter permease,718071
4,MFS transporter,608034


In [4]:
semantic_annotation_matches = rank_texts(
    annotation_counts_df["annotation"],
    "antimicrobial resistance not antibiotic synthesis",
    method="semantic",
    threshold=0.25,
    output_csv=ARTIFACT_OUTPUT_DIR / "semantic_annotation_matches.csv",
)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Batches:   0%|          | 0/483 [00:00<?, ?it/s]

In [5]:
semantic_annotation_matches_df = pd.read_csv(ARTIFACT_OUTPUT_DIR / "semantic_annotation_matches.csv")
display(semantic_annotation_matches_df.head())

,annotation,score,semantic_rank
0,resistance to heterologous antibiotics,0.60,1
1,antibiotic resistance protein,0.57,2
2,antibiotic-resistance protein,0.56,3
3,multiple antibiotic resistance protein,0.52,4
4,peptide antibiotic resistance protein,0.50,5


In [7]:
developer_prompt = """
You are classifying biological annotations for antimicrobial resistance (AMR).

AMR includes:
- antibiotic resistance proteins
- efflux pumps specific to antibiotics
- beta-lactamases
- target modification/protection
- antimicrobial peptide resistance

NOT AMR:
- antibiotic biosynthesis
- metabolism
- virulence
- metal resistance
- generic transport unless clearly resistance-related

Examples:
Annotation: methicillin resistance factor FemA
Category: antibiotic_resistance
is_amr: true

Annotation: phenazine antibiotic biosynthesis protein
Category: non_amr_antibiotic_biosynthesis
is_amr: false

Annotation: antibiotic ABC transporter
Category: ambiguous
is_amr: false
"""


class AMRLabel(BaseModel):
    annotation: str
    is_amr: bool
    category: Literal[
        "antibiotic_resistance",
        "non_amr_antibiotic_biosynthesis",
        "non_amr_transport",
        "non_amr_metal_resistance",
        "non_amr_virulence",
        "ambiguous",
        "unknown",
    ]
    confidence: float = Field(ge=0.0, le=1.0)
    evidence_terms: List[str]
    reason: str


In [ ]:
batch_size = 5
output_csv = ARTIFACT_OUTPUT_DIR / "semantic_annotation_amr_labels.csv"

source_df = semantic_annotation_matches_df.reset_index().rename(columns={"index": "row_idx"})

if output_csv.exists():
    saved_results_df = pd.read_csv(output_csv)
    completed_row_idxs = set(saved_results_df["row_idx"].tolist())
else:
    saved_results_df = pd.DataFrame()
    completed_row_idxs = set()

pending_df = source_df[~source_df["row_idx"].isin(completed_row_idxs)].copy()
results_batches = []

for batch_start in tqdm(range(0, len(pending_df), batch_size)):
    batch_df = pending_df.iloc[batch_start : batch_start + batch_size]
    batch_results = []

    try:
        for _, row in batch_df.iterrows():
            annotation = row["annotation"]

            response = client.responses.parse(
                model="gpt-5.4-mini",
                input=[
                    {"role": "system", "content": developer_prompt},
                    {"role": "user", "content": f"Classify this annotation: {annotation}"},
                ],
                text_format=AMRLabel,
            )

            parsed = response.output_parsed.model_dump()
            parsed["row_idx"] = row["row_idx"]
            batch_results.append(parsed)

        batch_results_df = pd.DataFrame(batch_results)
        results_batches.append(batch_results_df)
        saved_results_df = pd.concat([saved_results_df, batch_results_df], ignore_index=True)
        saved_results_df.to_csv(output_csv, index=False)

        print(f"Saved batch {batch_start // batch_size + 1} ({len(batch_results_df)} rows) to {output_csv.name}")
    except Exception as exc:
        print(f"Stopped at batch {batch_start // batch_size + 1}. Previously saved results remain in {output_csv.name}. Error: {exc}")
        break

if output_csv.exists():
    results_df = pd.read_csv(output_csv)
else:
    results_df = pd.concat(results_batches, ignore_index=True) if results_batches else pd.DataFrame()

final_df = source_df.merge(
    results_df,
    on="row_idx",
    how="left",
    suffixes=("_original", "_pred"),
)

print(final_df)

In [8]:
final_df = pd.read_csv("/home/ubuntu/projects/biodata/DNA-BERT/results/protein_tokenization/artifacts/semantic_annotation_matches_checkpoint.csv")

/tmp/ipykernel_17457/264096064.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv("/home/ubuntu/projects/biodata/DNA-BERT/results/protein_tokenization/artifacts/semantic_annotation_matches_checkpoint.csv")


In [9]:
final_df[final_df["category"] == "antibiotic_resistance"].tail(10).iloc[:, :4]

,row_idx,annotation,score,semantic_rank
67809,67809,lipid A C4-phosphatase,0.06,67810
67812,67812,MULTISPECIES: RND family transporter,0.06,67813
68074,68074,23S rRNA m(3)Psi1915 pseudouridine methyltransferase,0.06,68075
68233,68233,"DNA gyrase inhibitor YacG, partial",0.06,68234
68286,68286,MULTISPECIES: 4-amino-4-deoxy-L-arabinose-phospho-UDP flippase,0.06,68287
68317,68317,"Imm50 family immunity protein, partial",0.06,68318
68412,68412,MULTISPECIES: LiaI-LiaF-like domain-containing protein,0.06,68413
68416,68416,LPXTG-anchored DUF1542 repeat protein FmtB,0.06,68417
68424,68424,MULTISPECIES: MurT ligase domain-containing protein,0.06,68425
68662,68662,MULTISPECIES: response regulator transcription factor VncR,0.06,68663


In [10]:
final_df.category.isna().sum()

np.int64(54500)

In [11]:
final_df.category.value_counts()

category
unknown                            50276
non_amr_virulence                   5710
non_amr_transport                   4877
antibiotic_resistance               3462
non_amr_antibiotic_biosynthesis     1737
non_amr_metal_resistance            1688
ambiguous                           1145
Name: count, dtype: int64

In [ ]:
final_df[final_df.category == "antibiotic_resistance"].iloc[-4:, :-4].head()

,row_idx,annotation,score,semantic_rank,status,is_amr,category,confidence,evidence_terms,reason
68412,68412,MULTISPECIES: LiaI-LiaF-like domain-containing protein,0.06,68413,completed,NaN,antibiotic_resistance,NaN,[],NaN
68416,68416,LPXTG-anchored DUF1542 repeat protein FmtB,0.06,68417,completed,NaN,antibiotic_resistance,NaN,[],NaN
68424,68424,MULTISPECIES: MurT ligase domain-containing protein,0.06,68425,completed,NaN,antibiotic_resistance,NaN,[],NaN
68662,68662,MULTISPECIES: response regulator transcription factor VncR,0.06,68663,completed,NaN,antibiotic_resistance,NaN,[],NaN


In [13]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123395 entries, 0 to 123394
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   row_idx         123395 non-null  int64  
 1   annotation      123395 non-null  object 
 2   score           123395 non-null  float64
 3   semantic_rank   123395 non-null  int64  
 4   status          123395 non-null  object 
 5   is_amr          0 non-null       float64
 6   category        68895 non-null   object 
 7   confidence      10737 non-null   float64
 8   evidence_terms  123395 non-null  object 
 9   reason          10737 non-null   object 
 10  error_message   0 non-null       float64
 11  retry_count     123395 non-null  int64  
 12  completed_at    68895 non-null   object 
 13  updated_at      68895 non-null   object 
dtypes: float64(4), int64(3), object(7)
memory usage: 13.2+ MB
